In [24]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from scipy.stats import ks_2samp

# -----------------------
# Cargar datos
# -----------------------
train = pd.read_csv("train (1).csv")
test = pd.read_csv("test.csv")

# -----------------------
# Detectar target
# -----------------------
possible_targets = ["fail", "failed", "failure", "target", "label", "y"]
target_col = None

lower_map = {c.lower(): c for c in train.columns}
for t in possible_targets:
    if t in lower_map:
        target_col = lower_map[t]
        break

if target_col is None:
    raise ValueError("Define manualmente la columna objetivo.")

y = train[target_col]
X = train.drop(columns=[target_col])

# -----------------------
# Columnas categóricas / numéricas
# -----------------------
cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ]), cat_cols)
])

model = HistGradientBoostingClassifier(random_state=42)

clf = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

# -----------------------
# TASK 1 — Validación cruzada
# -----------------------
if "product_code" in X.columns:
    groups = X["product_code"].astype(str)
    cv = GroupKFold(n_splits=5)
    splits = cv.split(X, y, groups)
else:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    splits = cv.split(X, y)

oof = np.zeros(len(X))

for fold, (tr, va) in enumerate(splits, 1):
    clf.fit(X.iloc[tr], y.iloc[tr])
    preds = clf.predict_proba(X.iloc[va])[:, 1]
    oof[va] = preds
    auc = roc_auc_score(y.iloc[va], preds)
    print(f"Fold {fold} AUC: {auc:.4f}")

print("Overall AUC:", roc_auc_score(y, oof))

# -----------------------
# Entrenar modelo final
# -----------------------
clf.fit(X, y)

# Predicción test
probs = clf.predict_proba(test)[:, 1]

pd.DataFrame({
    "failure": probs
}).to_csv("Predictions.csv", index=False)

print("\nPredictions.csv generado.")

# -------------------------------------
# TASK 2 - Evaluación de Data Drift
# -------------------------------------

print("\n--- TASK 2: ADVERSARIAL DRIFT TEST ---")

train_adv = train.drop(columns=[target_col])
test_adv = test.copy()

if "id" in train_adv.columns:
    train_adv = train_adv.drop(columns=["id"])
    test_adv = test_adv.drop(columns=["id"])

train_adv["dataset"] = 0
test_adv["dataset"] = 1

combined = pd.concat([train_adv, test_adv], axis=0, ignore_index=True)

y_drift = combined["dataset"]
X_drift = combined.drop(columns=["dataset"])

cat_cols_drift = [c for c in X_drift.columns if X_drift[c].dtype == "object"]
num_cols_drift = [c for c in X_drift.columns if c not in cat_cols_drift]

preprocess_drift = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols_drift),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols_drift)
])

drift_model = Pipeline([
    ("prep", preprocess_drift),
    ("model", HistGradientBoostingClassifier(random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_drift = np.zeros(len(X_drift))

for tr, va in cv.split(X_drift, y_drift):
    drift_model.fit(X_drift.iloc[tr], y_drift.iloc[tr])
    oof_drift[va] = drift_model.predict_proba(X_drift.iloc[va])[:, 1]

drift_auc = roc_auc_score(y_drift, oof_drift)

print("Adversarial AUC:", drift_auc)

Fold 1 AUC: 0.5705
Fold 2 AUC: 0.5785
Fold 3 AUC: 0.5707
Fold 4 AUC: 0.5667
Fold 5 AUC: 0.5742
Overall AUC: 0.5715284210436535

Predictions.csv generado.

--- TASK 2: ADVERSARIAL DRIFT TEST ---
Adversarial AUC: 1.0


## ✅ CONCLUSIONES

### TASK 1 — Predict Product Failures

El modelo alcanzó un **ROC-AUC ≈ 0.57** en validación cruzada, lo que indica una capacidad discriminativa superior al azar, aunque de magnitud moderada. Este resultado sugiere que las variables disponibles aportan información relevante para la predicción del fallo, si bien la relación entre los atributos medidos y el evento de failure presenta una señal limitada.  

Las predicciones generadas para el conjunto de test corresponden a la **probabilidad estimada de fallo**, lo cual es metodológicamente consistente con el enunciado, que solicita la estimación de la *likelihood* de failure para cada registro.

---

### TASK 2 — Evaluate Data Drift

El clasificador adversarial entrenado para distinguir entre observaciones de `train.csv` y `test.csv` obtuvo un **AUC = 1.0**, lo que implica una separabilidad perfecta entre ambos datasets. Este comportamiento constituye evidencia de que los conjuntos de datos no son homogéneos desde el punto de vista distributivo, ya que el modelo puede identificar patrones que permiten discriminar inequívocamente su procedencia.

Desde una perspectiva teórica, si ambos datasets fueran generados por el mismo proceso estadístico (ausencia de data drift), las observaciones deberían ser indistinguibles y el clasificador exhibiría un rendimiento equivalente al azar (**AUC ≈ 0.5**). En cambio, la existencia de drift introduce diferencias sistemáticas en las distribuciones de las variables, permitiendo al modelo aprender reglas de separación y, en consecuencia, alcanzar valores de AUC significativamente superiores.

---

### Interpretación de los resultados

El comportamiento del clasificador adversarial evidencia que las observaciones de `train.csv` y `test.csv` presentan diferencias sistemáticas en las variables de entrada. La capacidad del modelo para discriminar entre ambos conjuntos de datos sugiere que estos no comparten la misma estructura distributiva, lo que resulta consistente con la presencia de data drift.

En condiciones de homogeneidad, el clasificador no dispondría de información suficiente para distinguir la procedencia de los registros, y su rendimiento sería equivalente al azar. En consecuencia, el resultado observado constituye una señal empírica de divergencia entre datasets, indicando que las distribuciones subyacentes difieren de manera significativa.